# Train model_11 — WildJailbreak-trained PromptCop (BAGEL adaptability pilot)

Fine-tunes a new promptcop on `allenai/wildjailbreak`'s `train` config (262K
rows, cleanly disjoint from the `eval` config already used in the held-out
test set — no leakage risk to reason through).

Run top to bottom:
1. **Setup** — installs, config, the `USE_FULL_DATASET` toggle
2. **Load data**
3. **Quick throughput check** (optional but recommended) — times ~50 real
   train steps and extrapolates total time *before* you commit to the full
   run. Skip straight to Section 4 if you already know your numbers.
4. **Full training**
5. **Save** — `save_pretrained`, so this drops in as `model_11` using the
   exact same scoring code as every other promptcop, no special-casing.

After this finishes, copy the saved folder to `MODELS_DIRECTORY/model_11` on
Drive, then run the before/after comparison cells in the main router
notebook.

## 1. Setup

In [ ]:
!pip -q install --upgrade huggingface-hub datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 154.3 MB/s eta 0:00:00


In [ ]:
import time, random, warnings, os
warnings.filterwarnings('ignore')
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x

In [ ]:
!hf auth login --token ''

In [ ]:
# If running in Colab and you want the saved model to land straight on
# Drive, uncomment this and point SAVE_DIR at the Drive path directly.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODEL_NAME   = 'meta-llama/Llama-Prompt-Guard-2-86M'
MAX_LENGTH   = 512
BATCH_SIZE   = 8             # conservative start for an 8GB 3070 mobile; bump
                             # to 16 only after Section 3 runs clean at 8
FP16         = True
EPOCHS       = 3             # matches the paper's convention
LR           = 5e-6            # matches BAGEL_new_finetune.ipynb's train_model() default
WARMUP_RATIO = 0.06
SEED         = 42

USE_FULL_DATASET = False     # <-- TOGGLE: False = subsampled pilot, True = full 262K
SUBSAMPLE_SIZE    = 20_000   # only used when USE_FULL_DATASET is False

SAVE_DIR = ''      # copy this whole folder to MODELS_DIRECTORY/model_11 on Drive afterward
# ============================================================

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (no GPU found)')

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device: cuda | NVIDIA A100-SXM4-40GB


## 2. Load data — WildJailbreak `train` config

In [ ]:
ds = load_dataset("allenai/wildjailbreak", "train", delimiter="\t",
                  keep_default_na=False, split="train")
df = ds.to_pandas()

def get_prompt(row):
    return row['adversarial'] if str(row['data_type']).startswith('adversarial') else row['vanilla']

df['prompt'] = df.apply(get_prompt, axis=1)
label_map = {'vanilla_harmful': 1, 'adversarial_harmful': 1, 'vanilla_benign': 0, 'adversarial_benign': 0}
df['label'] = df['data_type'].map(label_map)
df = df[['prompt', 'label']].dropna()
df['label'] = df['label'].astype(int)
print(f'WildJailbreak train config: {len(df):,} rows total')
print(df['label'].value_counts())

README.md:   0%|          | 0.00/16.2k [00:00<?, ?B/s]

train/train.tsv: reconstructing file:   0%|          |  0.00B /  531MB            

train/train.tsv: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

WildJailbreak train config: 261,559 rows total
label
1    132778
0    128781
Name: count, dtype: int64


In [ ]:
if not USE_FULL_DATASET:
    df_run = df.sample(n=min(SUBSAMPLE_SIZE, len(df)), random_state=SEED).reset_index(drop=True)
else:
    df_run = df

print(f'using {len(df_run):,} rows for this run  (USE_FULL_DATASET={USE_FULL_DATASET})')

texts = df_run['prompt'].tolist()
labels = df_run['label'].tolist()

using 20,000 rows for this run  (USE_FULL_DATASET=False)


## 3. Quick throughput check (recommended)

Times ~50 real train steps (forward + backward + optimizer, not just
inference) after a short warmup, then extrapolates to the full run. Takes
under a minute. Skip to Section 4 if you've already measured this GPU.

In [ ]:
# N_WARMUP, N_TIMED = 5, 50
# n_needed_for_bench = (N_WARMUP + N_TIMED) * BATCH_SIZE
# assert len(texts) >= n_needed_for_bench, \
#     f'need at least {n_needed_for_bench} rows for the benchmark, have {len(texts)}'

# _tok = AutoTokenizer.from_pretrained(MODEL_NAME)
# _model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# # Match train_model() exactly: fresh classifier head, not whatever PromptGuard-2 ships with
# _model.classifier = torch.nn.Linear(_model.classifier.in_features, 2)
# _model.num_labels = 2
# _model.to(device)
# _model.train()

# _enc = _tok(texts[:n_needed_for_bench], padding=True, truncation=True,
#            max_length=MAX_LENGTH, return_tensors='pt')
# _ds = TensorDataset(_enc['input_ids'], _enc['attention_mask'],
#                     torch.tensor(labels[:n_needed_for_bench], dtype=torch.long))
# _dl = DataLoader(_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
# assert len(_dl) >= N_WARMUP + N_TIMED, f'only {len(_dl)} batches available'

# _opt = torch.optim.AdamW(_model.parameters(), lr=LR)
# _it = iter(_dl)

# def _run_step():
#     ids, mask, lab = next(_it)
#     ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
#     outputs = _model(input_ids=ids, attention_mask=mask, labels=lab)
#     loss = outputs.loss
#     _opt.zero_grad()
#     loss.backward()
#     _opt.step()
#     return ids.size(0)

# print(f'warming up ({N_WARMUP} batches, untimed)...')
# for _ in range(N_WARMUP):
#     _run_step()
# if device == 'cuda':
#     torch.cuda.synchronize()

# print(f'timing {N_TIMED} batches...')
# _n_timed_samples = 0
# _t0 = time.perf_counter()
# for _ in range(N_TIMED):
#     _n_timed_samples += _run_step()
# if device == 'cuda':
#     torch.cuda.synchronize()
# _t1 = time.perf_counter()

# _samples_per_sec = _n_timed_samples / (_t1 - _t0)
# print(f'\n{_samples_per_sec:.2f} samples/sec')

# _target_rows = len(df) if USE_FULL_DATASET else SUBSAMPLE_SIZE
# _secs = (_target_rows * EPOCHS) / _samples_per_sec
# _h, _rem = divmod(_secs, 3600)
# _m, _ = divmod(_rem, 60)
# print(f'estimated time for the configured run ({_target_rows:,} rows x {EPOCHS} epochs): '
#      f'{int(_h)}h {int(_m)}m')

# print('\nfor comparison, other sizes at this throughput:')
# for n_rows in [5_000, 10_000, 20_000, 30_000, 50_000, len(df)]:
#     s = (n_rows * EPOCHS) / _samples_per_sec
#     tag = ' (full dataset)' if n_rows == len(df) else ''
#     print(f'  {n_rows:>7,} rows x {EPOCHS} epochs -> {s/3600:.2f} h{tag}')

# # free the benchmark model before the real training run below
# del _model, _opt
# if device == 'cuda':
#     torch.cuda.empty_cache()

## 4. Full training

Uses the same `texts`/`labels` built in Section 2 (respecting the
`USE_FULL_DATASET` toggle) — not the small benchmark slice from Section 3.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# Match train_model() exactly: fresh classifier head, not whatever PromptGuard-2 ships with
model.classifier = torch.nn.Linear(model.classifier.in_features, 2)
model.num_labels = 2
model.to(device)

enc = tok(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
ds_t = TensorDataset(enc['input_ids'], enc['attention_mask'], torch.tensor(labels, dtype=torch.long))
dl = DataLoader(ds_t, batch_size=BATCH_SIZE, shuffle=True)

# No scheduler, no AMP, no gradient clipping -- matches BAGEL_new_finetune.ipynb's
# train_model() exactly, so model_11 is trained the same way as models 1-8 and 10.
opt = torch.optim.AdamW(model.parameters(), lr=LR)

print(f'{len(dl)} batches/epoch x {EPOCHS} epochs = {len(dl)*EPOCHS} total steps')

config.json:   0%|          | 0.00/871 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

2500 batches/epoch x 3 epochs = 7500 total steps


In [ ]:
model.train()
t_start = time.perf_counter()
for ep in range(EPOCHS):
    total_loss = 0
    bar = tqdm(dl, desc=f'Epoch {ep + 1}')
    for ids, mask, lab in bar:
        ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
        outputs = model(input_ids=ids, attention_mask=mask, labels=lab)
        loss = outputs.loss

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f'Average loss in epoch {ep + 1}: {total_loss / len(dl):.6f}')

elapsed = time.perf_counter() - t_start
h, rem = divmod(elapsed, 3600)
m, s = divmod(rem, 60)
print(f'\ntraining finished in {int(h)}h {int(m)}m {int(s)}s')

Epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]

Average loss in epoch 1: 0.241522


Epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]

Average loss in epoch 2: 0.090633


Epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]

Average loss in epoch 3: 0.049053

training finished in 0h 32m 54s


## 5. Save

In [ ]:
SAVE_DIR = ''

model.eval()
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
print(f'saved to {SAVE_DIR}')
print('copy this folder to MODELS_DIRECTORY/model_11 on Drive, then run the')
print('before/after comparison cells in the main router notebook.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to <SAVE_DIR>
copy this folder to MODELS_DIRECTORY/model_11 on Drive, then run the
before/after comparison cells in the main router notebook.


In [ ]:
LOCAL_BACKUP = '/content/model_11_backup'
model.save_pretrained(LOCAL_BACKUP)
tok.save_pretrained(LOCAL_BACKUP)
print('safely backed up to', LOCAL_BACKUP)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

safely backed up to /content/model_11_backup
